# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nirajmalla123/ML_pipeline/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My ML task is **ranking/scoring**. I want to prioritize content pages for organic search performance review. The system will assign a priority score so an SEO or content team can decide which pages to investigate first. Ranking is appropriate because the team has limited time and needs a useful top-priority list, rather than just a yes/no prediction for every page.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/Nirajmalla123/ML_pipeline/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print("Task type: Ranking / scoring")
print("Dataset shape:", df.shape)


Task type: Ranking / scoring
Dataset shape: (30000, 44)


In [7]:
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_proxy,baseline_score
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1,0
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1,0
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1,0
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1,0


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

My target is a **proxy for pages showing a declining trend**. I define it as `trend_direction == "down"`. This label comes from an existing trend rule based on `trend_pct`, rather than an independent future outcome. Therefore, the task identifies pages associated with an existing declining trend; it does not prove that a page will decline in the future. I will not use `trend_direction` or `trend_pct` as model input features because that would leak the answer.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

df["is_declining_proxy"] = (
    df["trend_direction"].str.lower().eq("down").astype(int)
)

print("Pages:", len(df))
print("Declining proxy rate:", round(df["is_declining_proxy"].mean(), 3))
print("\nTrend direction counts:")
print(df["trend_direction"].value_counts(dropna=False))



Pages: 30000
Declining proxy rate: 0.542

Trend direction counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My primary metric is **Precision@50**. It measures the fraction of the 50 highest-ranked pages that have the declining-trend proxy label. This matches the practical decision of selecting 50 pages for review. A higher value means the shortlist contains more pages matching the proxy. I will compare the result against a simple hand-written baseline. Because the label is a proxy, this metric measures agreement with that label, not proven SEO improvement.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



def precision_at_k(scores, labels, k=50):
    scores = np.asarray(scores)
    labels = np.asarray(labels)
    top_k = np.argsort(-scores, kind="stable")[:k]
    return labels[top_k].mean()

# Simple baseline: stale pages that still receive impressions
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)

df["baseline_score"] = (
    stale * visible * df["impressions_90d"]
)

baseline_precision = precision_at_k(
    df["baseline_score"],
    df["is_declining_proxy"],
    k=50
)

print(f"Baseline Precision@50: {baseline_precision:.3f}")





Baseline Precision@50: 0.740


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row represents **one pseudonymized content item**, with its associated search-performance measurements and content characteristics. The starter dataset contains 30,000 rows across 32 pseudonymized clients. The measurements include impressions, clicks, CTR, average search position, content age, and update freshness. This unit allows me to rank individual content items for review without exposing client names or URLs.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Show the actual data and its unit of analysis
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("One row = one content item")

columns_to_show = [
    "search_volume",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "trend_direction"
]

display(df[columns_to_show].head(5))



Rows: 30000
Columns: 46
One row = one content item


,search_volume,impressions_90d,ctr,avg_position,content_age_days,days_since_last_update,trend_direction
0,10.0,3803,0.76,10.6,187,20,down
1,90.0,15320,0.05,20.3,445,25,down
2,0.0,12581,0.09,36.5,141,20,down
3,10.0,11751,0.49,6.2,463,22,stable
4,0.0,19140,0.13,44.0,263,14,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule can identify obvious cases, such as stale pages that still receive many impressions. However, organic search performance depends on several interacting signals, including visibility, CTR, search position, content age, and freshness. One threshold may miss pages that have different combinations of these signals. A model can learn more complex patterns and combine the signals into a ranking. It is not guaranteed to outperform a fixed rule, so I will test it against the baseline using the same metric and honest validation. The output is decision-support, not proof that a particular change will improve rankings.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Inspect how several numeric signals relate to the proxy label.
# These are exploratory associations, not causal effects.
features = [
    "search_volume",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "word_count"
]

available_features = [c for c in features if c in df.columns]

signal_correlations = (
    df[available_features + ["is_declining_proxy"]]
    .replace([np.inf, -np.inf], np.nan)
    .corr(numeric_only=True)["is_declining_proxy"]
    .drop("is_declining_proxy")
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

print("Absolute correlation with the declining-trend proxy:")
display(signal_correlations.to_frame("correlation").round(3))

print("\nBaseline Precision@50:", round(baseline_precision, 3))
print("This is a baseline, not proof that ML will perform better.")



Absolute correlation with the declining-trend proxy:


,correlation
content_age_days,-0.164
word_count,0.090
days_since_last_update,0.081
ctr,-0.062
avg_position,-0.029
search_volume,-0.019
impressions_90d,-0.018



Baseline Precision@50: 0.74
This is a baseline, not proof that ML will perform better.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.